In [47]:
import torch
import torchvision
import matplotlib.pyplot as plt
from tqdm import tqdm

In [48]:
devive = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [49]:
train_transforms = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=0.1736, std=0.3317),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
    torchvision.transforms.RandomVerticalFlip(p=0.5),
    torchvision.transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
])
test_transforms = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=0.1736, std=0.3317),
])

In [50]:
train = torchvision.datasets.EMNIST(root='./data', split='byclass', download=True, train=True, transform=train_transforms)
test = torchvision.datasets.EMNIST(root='./data', split='byclass', download=True, train=False, transform=test_transforms)

In [51]:
model = torchvision.models.resnet18()
model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.fc = torch.nn.Linear(512, 1)
model = model.to(devive)

In [56]:
train_loader = torch.utils.data.DataLoader(train, batch_size=1024, shuffle=True, num_workers=4, pin_memory=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)

loss = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

In [ ]:
EPOCHS = 20

for epoch in range(EPOCHS):
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    loss_epoch = 0.0
    for images, labels in tqdm(train_loader):
        images, labels = images.to(devive), labels.to(devive)
        preds = model(images).squeeze(1)
        loss_value = loss(preds, labels.float())
        loss_epoch += loss_value.item()
        optimizer.zero_grad()
        loss_value.backward()
        optimizer.step()

    print(f"Training Loss: {loss_epoch}")

    for test_images, test_labels in test_loader:
        test_images, test_labels = test_images.to(devive), test_labels.to(devive)
        test_preds = model(test_images).squeeze(1)
        test_loss_value = loss(test_preds, test_labels.float())
        print(f"Test Loss: {test_loss_value.item()}")

Epoch 1/20


 51%|█████████████████████████████████████████████████████████████████████████████████▌                                                                             | 350/682 [01:09<01:05,  5.10it/s]